## Exercise 0: Environment and libraries

In [1]:
import sys
print(sys.version)

3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


## Exercise 1: Reading files

In [4]:
with open("haiku.txt") as f:
    content = f.read()

print(content)

Old pond
Frog leaps in
Sound of water


In [6]:
with open("haiku.txt") as f:
    lines = f.readlines()
for line in lines:
    print(line, end="")

Old pond
Frog leaps in
Sound of water

In [8]:
with open("haiku.txt") as f:
    for line in f:
        print(line, end="")

Old pond
Frog leaps in
Sound of water

In [12]:
with open("haiku.txt") as f:
    for line in f:
        if "o" in line:
            print(line.strip())

Old pond
Frog leaps in
Sound of water


In [23]:
ans = ""
with open("haiku.txt", "rb") as f:
    data = f.read(9)
print(data)
print(data.decode("utf-8"))

b'Old pond\n'
Old pond



## Exercise 2: Writing and appending

In [13]:
from datetime import datetime

In [30]:
# with open("ex2.txt", "w") as f:
#     f.write("")

In [12]:
# with open("ex2.txt", "a") as f:
#     f.write("\nhello")

In [26]:
def write_log(filepath, message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(filepath, "a") as f:
        f.write(f"[{timestamp}] {message}\n")

In [31]:
write_log("ex2.txt", "Starting job")
write_log("ex2.txt", "Processing records")
write_log("ex2.txt", "Job complete")

In [32]:
with open("ex2.txt", "r") as f:
    print(f.read())

[2026-06-19 16:25:23] Starting job
[2026-06-19 16:25:23] Processing records
[2026-06-19 16:25:23] Job complete



In [39]:
with open("numbers.txt", "w") as f:
    f.write("")

In [40]:
with open("numbers.txt", "a") as f:
    for i in range(1, 11):
        f.write(f"{i}\n")

cont = open("numbers.txt")
print(cont.read())
cont.close()

1
2
3
4
5
6
7
8
9
10



In [44]:
with open("numbers.txt", "w") as f:
    for i in range(1, 11):
        if i % 2 == 0:
            f.write(f"{i}\n")
        else: continue

cont = open("numbers.txt")
print(cont.read())
cont.close()

2
4
6
8
10



## Exercise 3: Context managers

In [47]:
f = open("haiku.txt")
data = f.read()
print(f.closed)

False


In [50]:
with open("haiku.txt") as f:
    data = f.read()
print(f.closed)

True


In [51]:
import time

In [58]:
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
    def __exit__(self, exc_type, exc_val, exc_tb):
        dif = time.perf_counter() - self.start
        print(f"Elapsed: {dif:.4f} seconds")

In [59]:
with Timer():
    time.sleep(0.1)

Elapsed: 0.1006 seconds


In [63]:
from contextlib import contextmanager

In [65]:
@contextmanager
def timed_block():
    start = time.perf_counter()

    yield

    dif = time.perf_counter() - start
    print(f"Elapsed: {dif:.4f} seconds")

with timed_block():
    print("Hello World!!!")

Hello World!!!
Elapsed: 0.0003 seconds


## Exercise 4: Exceptions

In [67]:
def safe_divide(a, b):
    if b != 0:
        return a / b
    else:
        raise ValueError("Division by zero")

In [70]:
try:
    num = safe_divide(10, 0)
except Exception as e:
    print(f"Caught ValueError: {e}")
else:
    print(num)
finally:
    print("finally always runs")

Caught ValueError: Division by zero
finally always runs


In [79]:
class FileFormatError(ValueError):
    pass 

def parse_age(line):
    try:
        return int(line.strip())    
    except Exception:
        raise FileFormatError(f"Cannot parse age from: {line!r}")

tests = ["25", "not_a_number"]
for t in tests:
    try:
        print(parse_age(t))
    except Exception as e:
        print(e)

25
Cannot parse age from: 'not_a_number'


In [80]:
import json 

def load_config(path):
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as err:
        raise FileFormatError("Config file is not valid JSON") from err

In [81]:
# testing:
try:
    load_config("bad.json")
except FileFormatError as e:
    print(e)
    print(e.__cause__)

Config file is not valid JSON
Expecting property name enclosed in double quotes: line 2 column 5 (char 6)


## Exercise 5: CSV files

In [96]:
import csv

with open("products.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "name", "price", "stock"])
    writer.writerow([1, "Apple", 0.99, 120])
    writer.writerow([2, "Banana", 0.45, 300])
    writer.writerow([3, "Cherry", 2.50, 45])
    writer.writerow([4, "Date", 4.99, 8])

In [98]:
with open("products.csv") as f:
    print(f.read())

id,name,price,stock
1,Apple,0.99,120
2,Banana,0.45,300
3,Cherry,2.5,45
4,Date,4.99,8



In [99]:
with open("products.csv") as f:
    reader = csv.DictReader(f)

    for row in reader:
        if int(row["stock"]) > 50:
            print(row["name"], row["price"])

Apple 0.99
Banana 0.45


In [110]:
def apply_discount(input_path, output_path, discount_pct):
    with open(input_path, newline="") as f:
        reader = csv.DictReader(f)
        
        fieldnames = reader.fieldnames
        
        with open(output_path, "w", newline="") as of:
            writer = csv.DictWriter(of, fieldnames)

            writer.writeheader()
            for row in reader:
                price = float(row["price"])

                dprice = round(price*(1 - discount_pct/100), 2)

                row["price"] = dprice

                writer.writerow(row)                

In [112]:
apply_discount("products.csv", "discounted.csv", 10)
with open("discounted.csv", newline="") as f:
    # reader = csv.DictReader(f)
    # for row in reader:
    #     print(row)
    print(f.read())

id,name,price,stock
1,Apple,0.89,120
2,Banana,0.41,300
3,Cherry,2.25,45
4,Date,4.49,8



## Exercise 6: Paths with pathlib

In [113]:
from pathlib import Path

In [135]:
def describe_path(p):
    path = Path(p)
    # print(path)
    print(path.exists())
    if path.is_file():
        print("file")
        print(path.stem)
        print(path.suffix)
    elif path.is_dir():
        print("directory")
    else:
        print("None of them")

In [136]:
describe_path("haiku.txt")
print()
describe_path(".")

True
file
haiku
.txt

True
directory


In [150]:
files = []
def list_py_files(directory):
    path = Path(directory)
    items = path.iterdir()
    for i in items:
        if i.suffix in [".py", ".ipynb"]:
            files.append(i)
    return sorted(files)

In [151]:
for file in list_py_files("."):
    print(file)

Day04.ipynb


In [154]:
path = Path("output") / "reports" / "2024" / "summary.csv"
print(path)
print(path.parent)
path.parent.mkdir(parents=True, exist_ok=True)
path.touch()
print(path.exists())

output\reports\2024\summary.csv
output\reports\2024
True
